In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/SimoRinaldi/crop-spatial-classification.git
    else:
        !cd {REPO_DIR} && git pull
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    !pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato!")
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

In [ ]:
import os
import glob
import pandas as pd
import rasterio
import numpy as np
import concurrent.futures
from pathlib import Path
from tqdm.auto import tqdm

ground_truth = pd.read_json(f"{DATA_DIR}/interim/points.json")
path_sentinel2_data = Path(f"{DATA_DIR}/processed/sentinel2_data")

def process_single_point(args):
    index, row_dict, path_data_str = args
    crop_id = row_dict["code"]
    
    matching_dirs = glob.glob(os.path.join(path_data_str, f"point_{index}"))
    if not matching_dirs or not os.path.isdir(matching_dirs[0]):
        return index, None
        
    path_point_data = matching_dirs[0]

    band_means = {}

    # Priorità 1: Cerca il file unico mergiato sentinel2_data_*.tif
    merged_files = glob.glob(os.path.join(path_point_data, "sentinel2_data_*.tif"))
    if merged_files:
        try:
            with rasterio.open(merged_files[0]) as src:
                data = src.read()  # Shape: (n_layers, height, width)
                n_layers = src.count

                if n_layers >= 6:
                    b02_vals = [data[i].mean() for i in range(0, n_layers, 6)]
                    b03_vals = [data[i].mean() for i in range(1, n_layers, 6)]
                    b04_vals = [data[i].mean() for i in range(2, n_layers, 6)]
                    b08_vals = [data[i].mean() for i in range(3, n_layers, 6)]
                    b11_vals = [data[i].mean() for i in range(4, n_layers, 6)]
                    b12_vals = [data[i].mean() for i in range(5, n_layers, 6)]

                    band_means = {
                        "blu": float(np.mean(b02_vals)),
                        "verde": float(np.mean(b03_vals)),
                        "rosso": float(np.mean(b04_vals)),
                        "nir": float(np.mean(b08_vals)),
                        "swir1": float(np.mean(b11_vals)),
                        "swir2": float(np.mean(b12_vals)),
                    }
        except Exception:
            pass

    # Fallback (Priorità 2): Se il file mergiato non c'è o fallisce, usa i file TIF singoli
    if not band_means:
        def estrai_media_banda(codice_banda):
            files = [f for f in glob.glob(os.path.join(path_point_data, f"*{codice_banda}*.tif")) 
                     if not os.path.basename(f).startswith("sentinel2_data_")]
            valori_pixel = []
            for f in files:
                try:
                    with rasterio.open(f) as src:
                        valori_pixel.append(src.read().mean())
                except Exception:
                    pass
            return float(np.mean(valori_pixel)) if valori_pixel else 0.0

        band_means = {
            "blu": estrai_media_banda("_B02_"),
            "verde": estrai_media_banda("_B03_"),
            "rosso": estrai_media_banda("_B04_"),
            "nir": estrai_media_banda("_B08_"),
            "swir1": estrai_media_banda("_B11_"),
            "swir2": estrai_media_banda("_B12_"),
        }

    media_blu = band_means["blu"]
    media_verde = band_means["verde"]
    media_rosso = band_means["rosso"]
    media_nir = band_means["nir"]
    media_swir1 = band_means["swir1"]
    media_swir2 = band_means["swir2"]

    denominatore_ndvi = (media_nir + media_rosso)
    ndvi = (media_nir - media_rosso) / denominatore_ndvi if denominatore_ndvi > 0 else 0.0

    denominatore_ndwi = (media_nir + media_swir1)
    ndwi = (media_nir - media_swir1) / denominatore_ndwi if denominatore_ndwi > 0 else 0.0

    if media_blu > 0:
        return index, {
            "ID_Campo": index,
            "Ground_Truth": crop_id,
            "Blu_B02": media_blu,
            "Verde_B03": media_verde,
            "Rosso_B04": media_rosso,
            "NIR_B08": media_nir,
            "SWIR1_B11": media_swir1,
            "SWIR2_B12": media_swir2,
            "NDVI": ndvi,
            "NDWI": ndwi,
        }
    return index, None

# Preparazione dei task
tasks = [(index, row.to_dict(), str(path_sentinel2_data)) for index, row in ground_truth.iterrows()]

# Array pre-allocato per garantire l'associazione rigorosa dell'i-esimo elemento
results = [None] * len(tasks)
max_workers = min(32, os.cpu_count() or 4)

print(f"Avvio estrazione parallelizzata su {len(tasks)} campi con {max_workers} worker CPU...")

with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_single_point, t) for t in tasks]
    
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Elaborazione campi"):
        idx, res = future.result()
        if res is not None:
            results[idx] = res # Posizionamento ordinato nell'indice i-esimo

# Creazione del DataFrame finale preservando l'ordine dei punti
dataset = [r for r in results if r is not None]
df_finale = pd.DataFrame(dataset)
print(f"✅ Estrazione completata! Tabella ML creata con {len(df_finale)} campi.")


In [ ]:
processed_dir = DATA_DIR / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

# Salvataggio in Parquet
parquet_path = processed_dir / 'dataset_ml.parquet'
df_finale.to_parquet(parquet_path, index=False)

# Salvataggio in CSV
csv_path = processed_dir / 'dataset_ml.csv'
df_finale.to_csv(csv_path, index=False)

print(f"✅ Dataset salvato con successo sia in Parquet ({parquet_path}) che in CSV ({csv_path})!")
